# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://kaanismen.com")
links

['#main',
 '#top',
 '#about',
 '#skills',
 '#projects',
 '#experience',
 '#education',
 '#contact',
 'https://github.com/kaanismen',
 '#about',
 '#skills',
 '#projects',
 '#experience',
 '#education',
 '#contact',
 '#projects',
 'mailto:ismenkaan2003@gmail.com',
 '#about',
 'assets/report.pdf',
 'https://github.com/kaanismen/ExpenseApprovalApp',
 'mailto:ismenkaan2003@gmail.com',
 'https://github.com/kaanismen',
 'https://www.linkedin.com/in/kaanismen']

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://kaanismen.com"))


Here is the list of links on the website https://kaanismen.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main
#top
#about
#skills
#projects
#experience
#education
#contact
https://github.com/kaanismen
#about
#skills
#projects
#experience
#education
#contact
#projects
mailto:ismenkaan2003@gmail.com
#about
assets/report.pdf
https://github.com/kaanismen/ExpenseApprovalApp
mailto:ismenkaan2003@gmail.com
https://github.com/kaanismen
https://www.linkedin.com/in/kaanismen


In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [14]:
select_relevant_links("https://amiralturgutreistekneturu.com")

{'error': 'I didn’t receive the list of links. Please paste the full set of links found on the page (including any relative paths). I will convert relative links to full https URLs, filter out Terms of Service, Privacy, and email links, and return only the most relevant pages (e.g., About, Company/Corporate, Careers) in the requested JSON format.'}

In [15]:
select_relevant_links("https://kaanismen.com")

{'links': [{'type': 'about page', 'url': 'https://kaanismen.com/#about'},
  {'type': 'projects page', 'url': 'https://kaanismen.com/#projects'},
  {'type': 'experience page', 'url': 'https://kaanismen.com/#experience'},
  {'type': 'education page', 'url': 'https://kaanismen.com/#education'},
  {'type': 'contact page', 'url': 'https://kaanismen.com/#contact'},
  {'type': 'GitHub profile', 'url': 'https://github.com/kaanismen'},
  {'type': 'GitHub project',
   'url': 'https://github.com/kaanismen/ExpenseApprovalApp'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/kaanismen'},
  {'type': 'Portfolio PDF', 'url': 'https://kaanismen.com/assets/report.pdf'}]}

In [16]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [17]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [18]:
select_relevant_links("https://kaanismen.com")

Selecting relevant links for https://kaanismen.com by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'GitHub profile', 'url': 'https://github.com/kaanismen'},
  {'type': 'GitHub project',
   'url': 'https://github.com/kaanismen/ExpenseApprovalApp'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/kaanismen'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [19]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [20]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
baidu/Unlimited-OCR
Updated
1 day ago
•
363k
•
1.34k
zai-org/GLM-5.2
Updated
7 days ago
•
133k
•
2.9k
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
about 23 hours ago
•
908k
•
892
deepreinforce-ai/Ornith-1.0-35B-GGUF
Updated
4 days ago
•
124k
•
449
yuxinlu1/gemma-4-12B-agentic-fable5-com

In [29]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a company website
 and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
 Respond in markdown without code blocks.
 Include details of company culture, customers and careers/jobs if you have the information.
 """


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nbaidu/Unlimited-OCR\nUpdated\n1 day ago\n•\n363k\n•\n1.34k\nzai-org/GLM-5.2\nUpdated\n7 days ago\n•\n133k\n•\n2.9k\nempero-ai/

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("Kaan Ismen", "https://kaanismen.com")

Selecting relevant links for https://kaanismen.com by calling gpt-5-nano
Found 8 relevant links


# Kaan Ümit İşmen — Junior .NET Developer | Aspiring AI/ML Engineer

## About Kaan Ümit İşmen
Kaan Ümit İşmen is a dynamic and motivated Junior .NET Developer and aspiring AI/ML Engineer currently finalizing his B.Sc. in Computer Engineering with a specialization in Artificial Intelligence at Özyeğin University, expected graduation in January 2026.

With a unique blend of research and production experience, Kaan bridges the gap between cutting-edge AI research and practical software engineering. He is proficient in building generative models using PyTorch and delivering robust production code in C# and .NET environments.

## Technical Expertise & Skills
- Programming Languages & Frameworks:  
  C#, Python, Java, TypeScript

- Frameworks & Tools:  
  .NET, ASP.NET Core, Angular, EF Core, Unity

- AI/ML & Computer Vision:  
  GANs (CycleGAN), PyTorch, OpenCV, Computer Vision techniques

- Databases & DevOps:  
  SQL Server, Git for version control

- Other Skills:  
  Creating and managing deep learning models, especially in generative AI, prototyping models, and shipping scalable AI-powered applications.

## Key Projects & Achievements
- **Senior Project:** Fine-tuned a custom CycleGAN-Turbo model for pixel-art style transfer using a dataset of 6,000 unpaired images, enhancing understanding of generative models and scalable deep learning training.
- Completed over 10 GAN training experiments, comparing model performances to optimize results.
- Experience shipping production code in fintech environments (VeriPark) and developing interactive educational games using Unity (Wroftech).

## Career Interests
Kaan is eagerly looking for internship or full-time roles starting early 2026 that focus on AI/ML engineering, computer vision, and .NET software development, where he can leverage his academic knowledge and practical experience.

## Company Culture & Values
While Kaan Ismen is an individual professional rather than a company, his online presence reflects key personal and professional values:
- **Innovation:** Passionate about bringing research ideas into viable software solutions.
- **Continuous Learning:** Actively pursuing specialization in AI and related fields.
- **Quality & Deliverables:** Committed to shipping polished and efficient code.
- **Collaboration:** Experienced working across academia and industry roles.

## Contact & Availability
- Email: ismenkaan2003@gmail.com
- Open to hiring opportunities beginning early 2026.

---

*This brochure summarizes the professional profile of Kaan Ümit İşmen, a promising Junior .NET Developer and AI/ML enthusiast. Ideal for recruiters, collaborators, and tech companies seeking emerging talent skilled at the intersection of AI research and software production.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [26]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("Kaan Ismen", "https://kaanismen.com")

Selecting relevant links for https://kaanismen.com by calling gpt-5-nano
Found 7 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Kaan Ümit İşmen

**Junior .NET Developer | Aspiring AI/ML Engineer**

---

## About

Kaan Ümit İşmen is a dynamic Computer Engineer from Özyeğin University with a specialization in Artificial Intelligence, graduating in January 2026. Kaan stands at the intersection of research and production, uniquely skilled at bridging the gap between cutting-edge AI research and practical software development.

His senior project involved fine-tuning CycleGAN-Turbo for pixel-art style transfer on a 6,000-image unpaired dataset, demonstrating deep expertise in generative models and the complexities of training large-scale deep neural networks.

Kaan has hands-on experience shipping production code using C#, ASP.NET Core, and Angular during his tenure at VeriPark, a fintech company. He also revamped a Unity educational game at Wroftech, showcasing his versatility across different technologies and domains.

He is actively seeking opportunities—internship or full-time—starting early 2026, in AI/ML engineering, computer vision, and .NET development roles.

---

## Skills & Technologies

- **Programming & Frameworks:** C#, .NET, ASP.NET Core, Angular, TypeScript, Python, Java, SQL Server, Entity Framework Core  
- **AI & Machine Learning:** PyTorch, Generative Adversarial Networks (GANs), Computer Vision, CycleGAN  
- **Tools & Platforms:** Unity, OpenCV, Git  
- **Other:** Strong background in research and production workflows, with experience in both prototyping AI models and deploying them as scalable services.

---

## Key Projects

- **CycleGAN-Turbo Pixel-Art Style Transfer**  
  Developed and fine-tuned a generative model trained on a 6,000-image unpaired dataset to perform style transfer, refining expertise in generative models and large-scale training workflows.

- **Production Software at VeriPark**  
  Delivered production-ready features in fintech applications using C#, ASP.NET Core, and Angular that power real-world financial services.

- **Unity Educational Game Revamp at Wroftech**  
  Enhanced and rebuilt an educational game using Unity, blending software development and interactive media.

---

## Experience Summary

- **VeriPark (Fintech)**  
  Developed backend and frontend components, shipping robust software used in financial technology.

- **Wroftech**  
  Rebuilt and enhanced educational game projects using Unity.

- **Academic Research and Projects**  
  Ran over 10+ GAN training experiments to explore model architectures and improve results in generative AI.

---

## Company Culture / Professional Approach

Kaan embodies a culture of continuous learning and cross-disciplinary collaboration, combining academic rigor with practical software engineering skills. Open to innovation, Kaan thrives in environments where advanced research meets real-world application, focusing on impactful AI and software solutions.

---

## Contact & Availability

- **Email:** ismenkaan2003@gmail.com  
- **Open for:** Internship and full-time AI/ML engineering, computer vision, and .NET developer roles starting early 2026.

---

**Kaan Ümit İşmen — Bridging the future of AI research with scalable, production-ready software.**

In [30]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Kaan Ismen", "https://kaanismen.com")

Selecting relevant links for https://kaanismen.com by calling gpt-5-nano
Found 3 relevant links


# Meet Kaan Ümit İşmen: Your Future .NET Dev & Aspiring AI Wizard

---

## Who is Kaan?

Ever met a coder who’s part AI researcher, part full-stack software ninja?  
Meet **Kaan Ümit İşmen**, a Junior .NET Developer with a shiny B.Sc. in Computer Engineering (specialized in AI) from Özyeğin University, class of 2026 — but who’s already coding at pro level.

He’s the kind of engineer who can prototype cutting-edge generative models in Python with PyTorch *and* then ship rock-solid production apps in C#, ASP.NET Core, and Angular. Yes, bridging the science-to-software gap like a champ.

---

## Skills that Impress

- **AI/ML Mastery**: Plays with GANs like fine art — created and fine-tuned CycleGAN-Turbo models on a custom 6,000-image pixel art dataset. (Because who said AI can’t have style?)
- **Full-Stack Flair**: C#, .NET, ASP.NET Core, Angular — the triple threat for backend, frontend and all things web.
- **Game Dev Groove**: Rebuilt a Unity educational game — because learning should be fun, and coding games even more so.
- **Data & Vision**: SQL Server, OpenCV, computer vision projects — pixels and databases all in one brain.
- **Languages Galore**: Python, Java, TypeScript, and git for version control that even the perfectionists envy.

---

## The Projects & Experience Playground

- **CycleGAN Turbo for Pixel Style Transfer:** 10+ GAN training runs later, Kaan tunes AI-generated pixel art that would make retro gamers weep tears of joy.
- **Fintech at VeriPark:** Shipping C# and ASP.NET code that probably helps keep your money moving — securely and smoothly.
- **Unity Game Rebuild at Wroftech:** Making educational games that don’t bore your brain off.
- **Open Source Contributor:** Check out his GitHub to peek behind the curtain and see code that dances between efficiency and elegance.

---

## Company Culture? Career? Life?

Think of Kaan as a fusion reactor of curiosity + creativity + code. Always learning, always experimenting, always shipping. Looking for roles (internship or full-time) in AI/ML engineering, computer vision, or .NET development starting early 2026.  

If your team needs:  
- A developer who actually *gets* the research papers and turns them into real apps  
- A collaborator who codes clean but mentors friendly  
- A tech enthusiast who laughs in the face of complex GAN trainings and says “bring it on”  

Then, congratulations! You just found your secret weapon.

---

## Wanna Work with Kaan?

Email: ismenkaan2003@gmail.com  
Curious about his code playground? Dive into his [GitHub kingdom](https://github.com/kaanismen).  
Gotta get that AI/ML edge? He’s your guy — ready to turn big data dreams into real software magic.

---

## Final Thought

Kaan Ümit İşmen isn’t just a developer. He’s the perfect cocktail of AI ambition and production pragmatism — shaken, stirred, and served with a side of pixel-perfect passion.  

Invite Kaan to your project, your team, your future. Because AI just got a whole lot friendlier — and faster to deploy.

---

*Because when you want your tech future coded by someone who’s already living it, think “Kaan İşmen.”*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>